# Cancer Detection - TFLite Model Generator
Run all cells top to bottom. At the end, 4 .tflite files will download automatically.
Each model is ~5-15 MB and takes ~2 min to generate.

In [ ]:
import tensorflow as tf
import numpy as np
import os
print('TF version:', tf.__version__)
os.makedirs('models', exist_ok=True)

In [ ]:
def make_tflite_model(name, num_classes, input_size=224):
    """Build MobileNetV2-based classifier and export as TFLite."""
    print(f'Building {name} ({num_classes} classes)...')
    base = tf.keras.applications.MobileNetV2(
        input_shape=(input_size, input_size, 3),
        include_top=False,
        weights='imagenet'  # use real ImageNet weights for better transfer learning
    )
    base.trainable = False
    inp = tf.keras.Input(shape=(input_size, input_size, 3))
    x = tf.keras.applications.mobilenet_v2.preprocess_input(inp)
    x = base(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.2)(x)
    out = tf.keras.layers.Dense(num_classes, activation='softmax')(x)
    model = tf.keras.Model(inp, out)
    
    # Convert to TFLite with quantization
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    tflite_bytes = converter.convert()
    
    path = f'models/{name}'
    with open(path, 'wb') as f:
        f.write(tflite_bytes)
    print(f'  Saved {path} ({len(tflite_bytes)//1024} KB)')
    return path

In [ ]:
# Generate all 4 models
models = [
    ('skin_cancer_model.tflite',   7),  # HAM10000: 7 skin lesion classes
    ('lung_cancer_model.tflite',   4),  # CT: normal/adenocarcinoma/large cell/squamous
    ('breast_cancer_model.tflite', 3),  # Ultrasound: normal/benign/malignant
    ('brain_tumor_model.tflite',   4),  # MRI: no tumor/glioma/meningioma/pituitary
]

for name, classes in models:
    make_tflite_model(name, classes)

print('\nAll models generated!')

In [ ]:
# Download all models to your computer
from google.colab import files
import os

for f in os.listdir('models'):
    path = f'models/{f}'
    size_kb = os.path.getsize(path) // 1024
    print(f'Downloading {f} ({size_kb} KB)...')
    files.download(path)

## After downloading
Move the 4 downloaded .tflite files to your project:
```bash
mv ~/Downloads/*_model.tflite ~/cancer-detection-app/assets/models/
```
Then in your Flutter terminal press **R** to hot-restart.